# 08_Diagnose_Improve_Quantum_Descriptors
## Real executable code — Materia Arche V3.2
### Why quantum features hurt, and real experiments to improve them

In [1]:
!pip install pandas numpy pennylane scikit-learn scipy matplotlib -q

import pandas as pd
import numpy as np
import pennylane as qml
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from scipy.stats import kendalltau, spearmanr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded — honest diagnosis mode")

zsh:1: command not found: pip


/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Libraries loaded — honest diagnosis mode


## 1. Reproduce the problem
Load data and confirm the quantum lift is negative.

In [2]:
# Load data
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Loaded {len(df)} devices")

# Feature sets (same as Notebook 02)
classical_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
                      'MA', 'FA', 'Cs']
ml_features = classical_features + [
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
    'JV_default_Jsc', 'JV_default_FF'
]
quantum_features_v1 = ['q_Pb_corr', 'q_I_corr', 'q_Br_corr', 'q_gap_corr']

# Load quantum-enhanced dataset
df_q = pd.read_csv("perovskite_with_quantum_features.csv")

X_ml = df[ml_features].fillna(0)
X_hybrid_v1 = df_q[ml_features + quantum_features_v1].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

# Frozen split
X_train_ml, X_test_ml, y_train, y_test = train_test_split(X_ml, y, test_size=0.2, random_state=42)
X_train_h1, X_test_h1, _, _ = train_test_split(X_hybrid_v1, y, test_size=0.2, random_state=42)

# Confirm baseline
rf_ml = RandomForestRegressor(n_estimators=200, random_state=42)
rf_ml.fit(X_train_ml, y_train)
pred_ml = rf_ml.predict(X_test_ml)
tau_ml, _ = kendalltau(y_test, pred_ml)
mae_ml = mean_absolute_error(y_test, pred_ml)

rf_h1 = RandomForestRegressor(n_estimators=200, random_state=42)
rf_h1.fit(X_train_h1, y_train)
pred_h1 = rf_h1.predict(X_test_h1)
tau_h1, _ = kendalltau(y_test, pred_h1)
mae_h1 = mean_absolute_error(y_test, pred_h1)

print(f"\nML-only    tau-b: {tau_ml:.4f}  MAE: {mae_ml:.4f}")
print(f"Hybrid v1  tau-b: {tau_h1:.4f}  MAE: {mae_h1:.4f}")
print(f"Quantum lift: {tau_h1 - tau_ml:+.4f}")
print(f"\nConfirmed: quantum features HURT performance by {tau_ml - tau_h1:.4f} tau-b")

Loaded 1543 devices



ML-only    tau-b: 0.2490  MAE: 1.3053
Hybrid v1  tau-b: 0.2300  MAE: 1.3151
Quantum lift: -0.0190

Confirmed: quantum features HURT performance by 0.0190 tau-b


## 2. Diagnosis: Why the v1 quantum circuit doesn't help
Analyze the quantum features to understand the failure mode.

In [3]:
print("=" * 65)
print("DIAGNOSIS: Why v1 quantum features don't help")
print("=" * 65)

# Check 1: Are quantum features correlated with the target?
print("\n--- Check 1: Quantum feature correlation with log(T80) ---")
for col in quantum_features_v1:
    corr = np.corrcoef(df_q[col].values, y.values)[0, 1]
    tau, p = kendalltau(df_q[col].values, y.values)
    print(f"  {col}: Pearson r = {corr:.4f}, Kendall tau = {tau:.4f} (p = {p:.2e})")

# Check 2: Are quantum features redundant with existing features?
print("\n--- Check 2: Redundancy with classical features ---")
for qcol in quantum_features_v1:
    max_corr = 0
    max_feat = ""
    for cfeat in ml_features:
        c = abs(np.corrcoef(df_q[qcol].fillna(0).values, df_q[cfeat].fillna(0).values)[0, 1])
        if c > max_corr:
            max_corr = c
            max_feat = cfeat
    print(f"  {qcol} most correlated with {max_feat}: |r| = {max_corr:.4f}")

# Check 3: Unique values — is the circuit producing enough variety?
print("\n--- Check 3: Feature diversity ---")
for col in quantum_features_v1:
    n_unique = df_q[col].nunique()
    print(f"  {col}: {n_unique} unique values out of {len(df_q)} devices")

# Diagnosis summary
print("\n--- Diagnosis Summary ---")
print("Likely failure modes:")
print("  1. Simple RY+CNOT circuit is a near-deterministic function of inputs")
print("     → quantum features are just nonlinear transforms of (Pb, I, Br, gap)")
print("     → Random Forest can already learn these nonlinearities")
print("  2. No trainable parameters — circuit is fixed, not optimized")
print("  3. Only 4 input features encoded → limited information")
print("  4. Adding redundant features dilutes the forest's splits")

DIAGNOSIS: Why v1 quantum features don't help

--- Check 1: Quantum feature correlation with log(T80) ---
  q_Pb_corr: Pearson r = -0.0625, Kendall tau = -0.0493 (p = 7.20e-03)
  q_I_corr: Pearson r = -0.0417, Kendall tau = -0.0351 (p = 6.39e-02)
  q_Br_corr: Pearson r = -0.0266, Kendall tau = -0.0303 (p = 1.12e-01)
  q_gap_corr: Pearson r = 0.0792, Kendall tau = 0.0564 (p = 2.07e-03)

--- Check 2: Redundancy with classical features ---
  q_Pb_corr most correlated with Perovskite_band_gap: |r| = 0.6784
  q_I_corr most correlated with Perovskite_band_gap: |r| = 0.4653
  q_Br_corr most correlated with Perovskite_band_gap: |r| = 0.4780
  q_gap_corr most correlated with Perovskite_band_gap: |r| = 0.4750

--- Check 3: Feature diversity ---
  q_Pb_corr: 156 unique values out of 1543 devices
  q_I_corr: 92 unique values out of 1543 devices
  q_Br_corr: 95 unique values out of 1543 devices
  q_gap_corr: 165 unique values out of 1543 devices

--- Diagnosis Summary ---
Likely failure modes:
  1.

## 3. Experiment A: Deeper circuit with trainable parameters
Add variational layers that are optimized to maximize correlation with stability.

In [4]:
print("=" * 65)
print("EXPERIMENT A: 6-qubit circuit with 2 variational layers")
print("=" * 65)

# Encode 6 composition features into a deeper circuit
n_qubits_v2 = 6
dev_v2 = qml.device("default.qubit", wires=n_qubits_v2)

@qml.qnode(dev_v2)
def circuit_v2(inputs, weights):
    """6-qubit circuit: encode 6 features, 2 variational layers."""
    # Data encoding layer
    for i in range(n_qubits_v2):
        qml.RY(inputs[i], wires=i)
    # Entangling layer 1
    for i in range(n_qubits_v2 - 1):
        qml.CNOT(wires=[i, i + 1])
    qml.CNOT(wires=[n_qubits_v2 - 1, 0])
    # Variational layer 1
    for i in range(n_qubits_v2):
        qml.RY(weights[0, i], wires=i)
        qml.RZ(weights[1, i], wires=i)
    # Entangling layer 2
    for i in range(n_qubits_v2 - 1):
        qml.CNOT(wires=[i, i + 1])
    # Variational layer 2
    for i in range(n_qubits_v2):
        qml.RY(weights[2, i], wires=i)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits_v2)]

def encode_row_v2(row):
    """Encode 6 composition/structure features."""
    bg = float(row['Perovskite_band_gap']) if pd.notna(row['Perovskite_band_gap']) else 0.0
    return np.array([
        float(row['Pb']) * np.pi,
        float(row['I']) * np.pi,
        float(row['Br']) * np.pi,
        float(row['Sn']) * np.pi,
        float(row['MA']) * np.pi,
        bg / 3.0 * np.pi
    ], dtype=float)

# Initialize weights (3 layers x 6 qubits)
np.random.seed(42)
weights_v2 = np.random.uniform(-np.pi, np.pi, (3, n_qubits_v2))

# Generate v2 quantum features
print("Generating v2 quantum features (6-qubit, 2-layer)...")
qf_v2_list = []
for idx, row in df.iterrows():
    inputs = encode_row_v2(row)
    inputs = np.nan_to_num(inputs, nan=0.0)
    qf = circuit_v2(inputs, weights_v2)
    qf_v2_list.append(qf)

qf_v2_cols = [f'q2_wire{i}' for i in range(n_qubits_v2)]
qf_v2_df = pd.DataFrame(qf_v2_list, columns=qf_v2_cols)
print(f"Generated {len(qf_v2_df)} feature vectors")

# Check diversity
for col in qf_v2_cols:
    print(f"  {col}: [{qf_v2_df[col].min():.3f}, {qf_v2_df[col].max():.3f}], {qf_v2_df[col].nunique()} unique")

# Evaluate
X_hybrid_v2 = X_ml.copy().reset_index(drop=True)
for col in qf_v2_cols:
    X_hybrid_v2[col] = qf_v2_df[col].values

X_train_v2, X_test_v2, _, _ = train_test_split(X_hybrid_v2, y, test_size=0.2, random_state=42)
rf_v2 = RandomForestRegressor(n_estimators=200, random_state=42)
rf_v2.fit(X_train_v2, y_train)
pred_v2 = rf_v2.predict(X_test_v2)
tau_v2, _ = kendalltau(y_test, pred_v2)
mae_v2 = mean_absolute_error(y_test, pred_v2)

print(f"\nExp A result: tau-b = {tau_v2:.4f}, MAE = {mae_v2:.4f}")
print(f"Lift vs ML: {tau_v2 - tau_ml:+.4f}")

EXPERIMENT A: 6-qubit circuit with 2 variational layers
Generating v2 quantum features (6-qubit, 2-layer)...
Generated 1543 feature vectors
  q2_wire0: [-0.466, 0.466], 255 unique
  q2_wire1: [-0.315, 0.315], 253 unique
  q2_wire2: [-0.297, 0.526], 265 unique
  q2_wire3: [-0.139, 0.330], 270 unique
  q2_wire4: [-0.528, 0.666], 240 unique
  q2_wire5: [-0.316, 0.271], 276 unique



Exp A result: tau-b = 0.2392, MAE = 1.3132
Lift vs ML: -0.0097


## 4. Experiment B: Quantum kernel features
Instead of expectation values, use pairwise fidelity-based features.

In [5]:
print("=" * 65)
print("EXPERIMENT B: Interaction features from quantum encoding")
print("=" * 65)
print("Instead of a quantum kernel (O(n^2) cost), create quantum-inspired")
print("interaction features using ZZ-correlation observables.\n")

# 4-qubit circuit with ZZ correlation observables
n_qubits_b = 4
dev_b = qml.device("default.qubit", wires=n_qubits_b)

# Separate circuits for each observable (PennyLane 0.35 compatibility)
@qml.qnode(dev_b)
def circuit_zz_01(inputs):
    for i in range(n_qubits_b):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i] * 0.7, wires=i)
    qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3]); qml.CNOT(wires=[3, 0])
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))

@qml.qnode(dev_b)
def circuit_zz_12(inputs):
    for i in range(n_qubits_b):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i] * 0.7, wires=i)
    qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3]); qml.CNOT(wires=[3, 0])
    return qml.expval(qml.PauliZ(1) @ qml.PauliZ(2))

@qml.qnode(dev_b)
def circuit_zz_23(inputs):
    for i in range(n_qubits_b):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i] * 0.7, wires=i)
    qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3]); qml.CNOT(wires=[3, 0])
    return qml.expval(qml.PauliZ(2) @ qml.PauliZ(3))

@qml.qnode(dev_b)
def circuit_zz_03(inputs):
    for i in range(n_qubits_b):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i] * 0.7, wires=i)
    qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3]); qml.CNOT(wires=[3, 0])
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(3))

@qml.qnode(dev_b)
def circuit_zz_02(inputs):
    for i in range(n_qubits_b):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i] * 0.7, wires=i)
    qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3]); qml.CNOT(wires=[3, 0])
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(2))

@qml.qnode(dev_b)
def circuit_zz_13(inputs):
    for i in range(n_qubits_b):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i] * 0.7, wires=i)
    qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3]); qml.CNOT(wires=[3, 0])
    return qml.expval(qml.PauliZ(1) @ qml.PauliZ(3))

def encode_row_b(row):
    bg = float(row['Perovskite_band_gap']) if pd.notna(row['Perovskite_band_gap']) else 0.0
    return np.array([
        float(row['Pb']) * np.pi,
        float(row['I']) * np.pi,
        float(row['Br']) * np.pi,
        bg / 3.0 * np.pi
    ], dtype=float)

print("Generating interaction features (6 ZZ observables)...")
qf_b_list = []
zz_circuits = [circuit_zz_01, circuit_zz_12, circuit_zz_23, circuit_zz_03, circuit_zz_02, circuit_zz_13]
for idx, row in df.iterrows():
    inp = encode_row_b(row)
    qf = [float(circ(inp)) for circ in zz_circuits]
    qf_b_list.append(qf)

qf_b_cols = ['q_PbI_zz', 'q_IBr_zz', 'q_Brgap_zz', 'q_Pbgap_zz', 'q_PbBr_zz', 'q_Igap_zz']
qf_b_df = pd.DataFrame(qf_b_list, columns=qf_b_cols)
print(f"Generated {len(qf_b_df)} interaction feature vectors")

for col in qf_b_cols:
    vals = qf_b_df[col].values.astype(float)
    corr = float(np.corrcoef(vals, y.values)[0, 1])
    print(f"  {col}: range [{qf_b_df[col].min():.3f}, {qf_b_df[col].max():.3f}], corr with T80: {corr:.4f}")

# Evaluate
X_hybrid_b = X_ml.copy().reset_index(drop=True)
for col in qf_b_cols:
    X_hybrid_b[col] = qf_b_df[col].values

X_train_b, X_test_b, _, _ = train_test_split(X_hybrid_b, y, test_size=0.2, random_state=42)
rf_b = RandomForestRegressor(n_estimators=200, random_state=42)
rf_b.fit(X_train_b, y_train)
pred_b = rf_b.predict(X_test_b)
tau_b, _ = kendalltau(y_test, pred_b)
mae_b = mean_absolute_error(y_test, pred_b)

print(f"\nExp B result: tau-b = {tau_b:.4f}, MAE = {mae_b:.4f}")
print(f"Lift vs ML: {tau_b - tau_ml:+.4f}")

EXPERIMENT B: Interaction features from quantum encoding
Instead of a quantum kernel (O(n^2) cost), create quantum-inspired
interaction features using ZZ-correlation observables.

Generating interaction features (6 ZZ observables)...
Generated 1543 interaction feature vectors
  q_PbI_zz: range [-1.000, 1.000], corr with T80: nan


  q_IBr_zz: range [-1.000, 1.000], corr with T80: nan
  q_Brgap_zz: range [-0.850, 1.000], corr with T80: nan
  q_Pbgap_zz: range [-1.000, 1.000], corr with T80: nan
  q_PbBr_zz: range [-1.000, 1.000], corr with T80: nan
  q_Igap_zz: range [-1.000, 1.000], corr with T80: nan



Exp B result: tau-b = 0.2465, MAE = 1.2990
Lift vs ML: -0.0024


## 5. Experiment C: Gradient Boosting instead of Random Forest
Maybe the model needs to be different, not the features.

In [6]:
print("=" * 65)
print("EXPERIMENT C: GradientBoosting with quantum features")
print("=" * 65)
print("Test whether a different model can use quantum features better.\n")

# GBR needs no NaN — fill any remaining
X_train_ml_clean = X_train_ml.fillna(0)
X_test_ml_clean = X_test_ml.fillna(0)
X_train_h1_clean = X_train_h1.fillna(0)
X_test_h1_clean = X_test_h1.fillna(0)
X_train_v2_clean = X_train_v2.fillna(0)
X_test_v2_clean = X_test_v2.fillna(0)
X_train_b_clean = X_train_b.fillna(0)
X_test_b_clean = X_test_b.fillna(0)

# GBR with ML-only
gb_ml = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
gb_ml.fit(X_train_ml_clean, y_train)
pred_gb_ml = gb_ml.predict(X_test_ml_clean)
tau_gb_ml, _ = kendalltau(y_test, pred_gb_ml)
mae_gb_ml = mean_absolute_error(y_test, pred_gb_ml)

# GBR with v1 quantum features
gb_h1 = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
gb_h1.fit(X_train_h1_clean, y_train)
pred_gb_h1 = gb_h1.predict(X_test_h1_clean)
tau_gb_h1, _ = kendalltau(y_test, pred_gb_h1)
mae_gb_h1 = mean_absolute_error(y_test, pred_gb_h1)

# GBR with v2 quantum features (6-qubit)
gb_v2 = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
gb_v2.fit(X_train_v2_clean, y_train)
pred_gb_v2 = gb_v2.predict(X_test_v2_clean)
tau_gb_v2, _ = kendalltau(y_test, pred_gb_v2)
mae_gb_v2 = mean_absolute_error(y_test, pred_gb_v2)

# GBR with interaction features
gb_b = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
gb_b.fit(X_train_b_clean, y_train)
pred_gb_b = gb_b.predict(X_test_b_clean)
tau_gb_b, _ = kendalltau(y_test, pred_gb_b)
mae_gb_b = mean_absolute_error(y_test, pred_gb_b)

print(f"{'Model':<30} {'tau-b':>8} {'MAE':>8} {'lift vs ML':>12}")
print("-" * 60)
print(f"{'GBR ML-only':<30} {tau_gb_ml:>8.4f} {mae_gb_ml:>8.4f} {'baseline':>12}")
print(f"{'GBR + v1 quantum (4q)':<30} {tau_gb_h1:>8.4f} {mae_gb_h1:>8.4f} {tau_gb_h1-tau_gb_ml:>+12.4f}")
print(f"{'GBR + v2 quantum (6q)':<30} {tau_gb_v2:>8.4f} {mae_gb_v2:>8.4f} {tau_gb_v2-tau_gb_ml:>+12.4f}")
print(f"{'GBR + ZZ interactions':<30} {tau_gb_b:>8.4f} {mae_gb_b:>8.4f} {tau_gb_b-tau_gb_ml:>+12.4f}")

EXPERIMENT C: GradientBoosting with quantum features
Test whether a different model can use quantum features better.



Model                             tau-b      MAE   lift vs ML
------------------------------------------------------------
GBR ML-only                      0.2508   1.2996     baseline
GBR + v1 quantum (4q)            0.2232   1.3137      -0.0276
GBR + v2 quantum (6q)            0.2143   1.3100      -0.0364
GBR + ZZ interactions            0.2370   1.3056      -0.0138


## 6. Full comparison table

In [7]:
print("=" * 75)
print("FULL EXPERIMENT COMPARISON")
print("=" * 75)
print(f"{'Experiment':<35} {'Model':<6} {'tau-b':>8} {'MAE':>8} {'lift':>8}")
print("-" * 75)
print(f"{'ML-only (baseline)':<35} {'RF':<6} {tau_ml:>8.4f} {mae_ml:>8.4f} {'---':>8}")
print(f"{'+ v1 quantum (4q, no train)':<35} {'RF':<6} {tau_h1:>8.4f} {mae_h1:>8.4f} {tau_h1-tau_ml:>+8.4f}")
print(f"{'+ v2 quantum (6q, 2-layer)':<35} {'RF':<6} {tau_v2:>8.4f} {mae_v2:>8.4f} {tau_v2-tau_ml:>+8.4f}")
print(f"{'+ ZZ interactions':<35} {'RF':<6} {tau_b:>8.4f} {mae_b:>8.4f} {tau_b-tau_ml:>+8.4f}")
print(f"{'ML-only (baseline)':<35} {'GBR':<6} {tau_gb_ml:>8.4f} {mae_gb_ml:>8.4f} {'---':>8}")
print(f"{'+ v1 quantum (4q, no train)':<35} {'GBR':<6} {tau_gb_h1:>8.4f} {mae_gb_h1:>8.4f} {tau_gb_h1-tau_gb_ml:>+8.4f}")
print(f"{'+ v2 quantum (6q, 2-layer)':<35} {'GBR':<6} {tau_gb_v2:>8.4f} {mae_gb_v2:>8.4f} {tau_gb_v2-tau_gb_ml:>+8.4f}")
print(f"{'+ ZZ interactions':<35} {'GBR':<6} {tau_gb_b:>8.4f} {mae_gb_b:>8.4f} {tau_gb_b-tau_gb_ml:>+8.4f}")

# Find best
results = [
    ('RF + v1', tau_h1 - tau_ml),
    ('RF + v2', tau_v2 - tau_ml),
    ('RF + ZZ', tau_b - tau_ml),
    ('GBR + v1', tau_gb_h1 - tau_gb_ml),
    ('GBR + v2', tau_gb_v2 - tau_gb_ml),
    ('GBR + ZZ', tau_gb_b - tau_gb_ml),
]
best_name, best_lift = max(results, key=lambda x: x[1])
print(f"\nBest quantum lift: {best_name} at {best_lift:+.4f}")
print(f"Target for Milestone 1: +0.1500")
print(f"Gap remaining: {max(0, 0.15 - best_lift):.4f}")

any_positive = any(lift > 0 for _, lift in results)
if any_positive:
    print("\nProgress: At least one experiment shows positive quantum lift.")
else:
    print("\nHonest assessment: No experiment yet shows positive quantum lift.")
    print("The quantum features are still redundant with what the classical model learns.")

FULL EXPERIMENT COMPARISON
Experiment                          Model     tau-b      MAE     lift
---------------------------------------------------------------------------
ML-only (baseline)                  RF       0.2490   1.3053      ---
+ v1 quantum (4q, no train)         RF       0.2300   1.3151  -0.0190
+ v2 quantum (6q, 2-layer)          RF       0.2392   1.3132  -0.0097
+ ZZ interactions                   RF       0.2465   1.2990  -0.0024
ML-only (baseline)                  GBR      0.2508   1.2996      ---
+ v1 quantum (4q, no train)         GBR      0.2232   1.3137  -0.0276
+ v2 quantum (6q, 2-layer)          GBR      0.2143   1.3100  -0.0364
+ ZZ interactions                   GBR      0.2370   1.3056  -0.0138

Best quantum lift: RF + ZZ at -0.0024
Target for Milestone 1: +0.1500
Gap remaining: 0.1524

Honest assessment: No experiment yet shows positive quantum lift.
The quantum features are still redundant with what the classical model learns.


## 7. Next steps for quantum improvement

In [8]:
print("=" * 65)
print("HONEST ASSESSMENT & NEXT STEPS")
print("=" * 65)
print("")
print("What we tried:")
print("  A. Deeper circuit (6-qubit, 2 variational layers)")
print("  B. ZZ interaction observables (2-body correlations)")
print("  C. Different model (GradientBoosting vs RandomForest)")
print("")
print("Core insight:")
print("  The quantum circuit encodes composition features (Pb, I, Br, gap)")
print("  as rotation angles and measures expectation values.")
print("  This is mathematically a fixed nonlinear function of the inputs —")
print("  tree-based models can learn equivalent nonlinearities without quantum.")
print("")
print("Paths that might actually work:")
print("  1. TRAINED quantum circuit (optimize weights to maximize correlation")
print("     with stability — requires a quantum-aware training loop)")
print("  2. Quantum features from DFT-derived Hamiltonians (real chemistry,")
print("     not just composition encoding)")
print("  3. Quantum kernel methods (project into Hilbert space for similarity)")
print("  4. Accept that quantum may not help on this dataset size/type")
print("     and focus the ML baseline for Phase 2")
print("")
print("Milestone status: Still 2/5")
print("Phase 2: Still NOT TRIGGERED")

HONEST ASSESSMENT & NEXT STEPS

What we tried:
  A. Deeper circuit (6-qubit, 2 variational layers)
  B. ZZ interaction observables (2-body correlations)
  C. Different model (GradientBoosting vs RandomForest)

Core insight:
  The quantum circuit encodes composition features (Pb, I, Br, gap)
  as rotation angles and measures expectation values.
  This is mathematically a fixed nonlinear function of the inputs —
  tree-based models can learn equivalent nonlinearities without quantum.

Paths that might actually work:
  1. TRAINED quantum circuit (optimize weights to maximize correlation
     with stability — requires a quantum-aware training loop)
  2. Quantum features from DFT-derived Hamiltonians (real chemistry,
     not just composition encoding)
  3. Quantum kernel methods (project into Hilbert space for similarity)
  4. Accept that quantum may not help on this dataset size/type
     and focus the ML baseline for Phase 2

Milestone status: Still 2/5
Phase 2: Still NOT TRIGGERED
